In [ ]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperNeuralOperator,
)
from context_flux_no.training.io import load_model
from context_flux_no.utils import num_parameters
from einops import rearrange
from jaxtyping import Array, Float, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
# TODO: make this more ergonomic
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FNO",
    target_network_kwargs=dict(
        frequency_modes=8, lift_dim=48, depth=4, width_lift=48, width_project=48
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="UNet",
    target_network_kwargs=dict(hidden_channels_base=8, groups_norm=4, stack_grid=True),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)
dummy_target = hyperfluxno.hypernetwork_head(
    hyperfluxno.hypernetwork_trunk(
        jnp.zeros(
            hyperfluxno.embedding_dim,
        )
    )
)
print(num_parameters(dummy_target) / 1e6)

In [ ]:
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), (512, 21, 1, 100))

In [ ]:
hyperfluxno(test_data[0, :20], (0.1, 0.01))

In [ ]:
loss_fn(hyperfluxno, test_data, (0.1, 0.01), jax.random.key(0))

## Evaluate the trained target networks

In [ ]:
model_paths = {
    "FNO": [
        "seed=0/26-07-10-01:51:12",
        "seed=10/26-07-10-01:34:31",
        "seed=20/26-07-10-01:34:31",
    ],
    "UNet": [
        "seed=0/26-07-10-01:39:05",
        "seed=10/26-07-10-01:39:05",
        "seed=20/26-07-10-01:39:05",
    ],
}

In [ ]:
datadir = Path("../../data")
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [ ]:
results_dict = defaultdict(list)
for target_type, paths in model_paths.items():
    for p in tqdm(paths):
        loaddir = Path(
            "../../checkpoints/cubicflux_1d/HyperNeuralOperator/OneStepLoss/"
        )
        model = load_model(loaddir / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[target_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    print(
        model_name,
        jax.tree.map(
            lambda list_: {
                "mean": jnp.mean(jnp.asarray(list_)),
                "std": jnp.std(jnp.asarray(list_)),
            },
            res_,
            is_leaf=lambda x: isinstance(x, list),
        ),
    )

In [ ]:
model_paths = {
    "ContextAppendedFluxNO": [
        "seed=0/26-08-01-21_27_02",
        "seed=10/26-08-01-21_27_02",
        "seed=20/26-08-01-21_27_02",
    ],
}